# Component 1 / Phase 4 — Overlap risk model (Colab)

**District:** Anuradhapura · **Grid:** 500 m · **CRS:** EPSG:5234

### Upload these two files first
1. `training_set.parquet`
2. `feature_grid.parquet`

Use the folder icon (left) → Upload, **or** put them on Google Drive and set `DATA_DIR` below.

### What this notebook does
1. Spatial cross-validation (XGBoost)
2. Fit final model → habitat suitability
3. Combine with human exposure → `risk_score`
4. Save model + scores + simple plots (download zip at the end)

In [ ]:
# Install (Colab)
!pip install -q xgboost shap scikit-learn pandas numpy pyarrow matplotlib joblib

In [ ]:
from pathlib import Path
import json
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier

# --- paths: change if you uploaded to Drive ---
# Option A: files uploaded to Colab session root / content
DATA_DIR = Path('/content')
# Option B: Google Drive example (uncomment after mounting):
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/hec-phase4')

OUT_DIR = Path('/content/phase4_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'training_set.parquet'
GRID_PATH = DATA_DIR / 'feature_grid.parquet'

# also search common upload locations
for p in [Path('/content'), Path('/content/data'), Path('/content/processed')]:
    if (p / 'training_set.parquet').exists():
        TRAIN_PATH = p / 'training_set.parquet'
        GRID_PATH = p / 'feature_grid.parquet'
        break

assert TRAIN_PATH.exists(), f'Missing {TRAIN_PATH} — upload training_set.parquet'
assert GRID_PATH.exists(), f'Missing {GRID_PATH} — upload feature_grid.parquet'
print('Train:', TRAIN_PATH)
print('Grid:', GRID_PATH)

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
grid = pd.read_parquet(GRID_PATH)
print('training_set', train.shape, 'label counts', train['label'].value_counts().to_dict())
print('feature_grid', grid.shape)

# Predictors for habitat suitability (exclude IDs / labels / meta / precomputed exposure)
# human_exposure_raw is applied AFTER prediction so we do not double-count people.
# `district` appears in pooled AN+PO packs — keep it for CV grouping only, never as a feature.
EXCLUDE = {
    'cell_id', 'ix', 'iy', 'cx', 'cy',
    'presence', 'n_records', 'label', 'sample_type',
    'human_exposure_raw', 'district',
}
FEATURE_COLS = [
    c for c in train.columns
    if c not in EXCLUDE and pd.api.types.is_numeric_dtype(train[c])
]
# Score grid must have the same columns (district is train-only)
missing = [c for c in FEATURE_COLS if c not in grid.columns]
if missing:
    raise ValueError(f'feature_grid missing columns: {missing}')
print('Features (' + str(len(FEATURE_COLS)) + '):', FEATURE_COLS)

X = train[FEATURE_COLS].copy()
y = train['label'].astype(int).to_numpy()

# Spatial block id for GroupKFold (≈ 10 km blocks on a 500 m grid)
BLOCK = 20  # 20 * 500 m = 10 km
groups = (train['ix'] // BLOCK).astype(str) + '_' + (train['iy'] // BLOCK).astype(str)
if 'district' in train.columns:
    groups = train['district'].astype(str) + '_' + groups
print('Spatial groups:', groups.nunique())

## 1. Spatial cross-validation

In [ ]:
def make_model(scale_pos_weight: float) -> XGBClassifier:
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective='binary:logistic',
        eval_metric='auc',
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
    )

spw = float((y == 0).sum() / max((y == 1).sum(), 1))
print('scale_pos_weight', round(spw, 2))

gkf = GroupKFold(n_splits=5)
fold_rows = []
oof = np.zeros(len(y), dtype=float)

for fold, (tr, te) in enumerate(gkf.split(X, y, groups), start=1):
    model = make_model(spw)
    model.fit(X.iloc[tr], y[tr])
    proba = model.predict_proba(X.iloc[te])[:, 1]
    oof[te] = proba
    auc = roc_auc_score(y[te], proba)
    ap = average_precision_score(y[te], proba)
    fold_rows.append({'fold': fold, 'auc': auc, 'average_precision': ap, 'n_test': len(te)})
    print(f'Fold {fold}: AUC={auc:.3f}  AP={ap:.3f}  n_test={len(te)}')

cv = pd.DataFrame(fold_rows)
print('\nMean AUC', round(float(cv['auc'].mean()), 3), '±', round(float(cv['auc'].std()), 3))
print('Mean AP ', round(float(cv['average_precision'].mean()), 3))
cv.to_csv(OUT_DIR / 'spatial_cv_metrics.csv', index=False)

overall_auc = roc_auc_score(y, oof)
overall_ap = average_precision_score(y, oof)
print('OOF AUC', round(overall_auc, 3), 'OOF AP', round(overall_ap, 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fpr, tpr, _ = roc_curve(y, oof)
ax[0].plot(fpr, tpr, label=f'OOF AUC={overall_auc:.3f}')
ax[0].plot([0, 1], [0, 1], '--', color='grey')
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].set_title('ROC (spatial OOF)')
ax[0].legend()

prec, rec, _ = precision_recall_curve(y, oof)
ax[1].plot(rec, prec, label=f'AP={overall_ap:.3f}')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('PR (spatial OOF)')
ax[1].legend()
plt.tight_layout()
fig.savefig(OUT_DIR / 'cv_roc_pr.png', dpi=150)
plt.show()

## 2. Final model + score full district

In [ ]:
final = make_model(spw)
final.fit(X, y)
joblib.dump(
    {'model': final, 'feature_cols': FEATURE_COLS, 'scale_pos_weight': spw, 'block': BLOCK},
    OUT_DIR / 'xgb_habitat_model.joblib',
)

# Habitat suitability on every cell
X_all = grid[FEATURE_COLS].copy()
habitat = final.predict_proba(X_all)[:, 1]

# Human exposure 0–1 from existing raw score
exp_raw = grid['human_exposure_raw'].to_numpy(dtype=float)
exp = exp_raw.copy()
finite = np.isfinite(exp)
if finite.any():
    lo, hi = np.nanpercentile(exp[finite], [2, 98])
    exp = np.clip((exp - lo) / (hi - lo + 1e-9), 0, 1)
else:
    exp = np.zeros_like(exp)

# Overlap risk (novelty): not forest alone — habitat × people
risk = habitat * exp
# optional mild floor so remote habitat is not exactly zero everywhere
# risk = 0.85 * (habitat * exp) + 0.15 * habitat

out = grid[['cell_id', 'ix', 'iy', 'cx', 'cy']].copy()
out['habitat_suitability'] = habitat
out['human_exposure'] = exp
out['risk_score'] = risk
out['risk_score_0_100'] = (100 * (risk - risk.min()) / (risk.max() - risk.min() + 1e-9)).round(2)

out.to_parquet(OUT_DIR / 'risk_scores.parquet', index=False)
out.to_csv(OUT_DIR / 'risk_scores.csv', index=False)
print(out[['habitat_suitability', 'human_exposure', 'risk_score']].describe())
print('Saved risk_scores.parquet / .csv')

## 3. Feature importance + SHAP (why a place is risky — Phase 5 preview)

In [ ]:
imp = pd.Series(final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
imp.to_csv(OUT_DIR / 'feature_importance.csv', header=['importance'])

fig, ax = plt.subplots(figsize=(7, 5))
imp.head(15).iloc[::-1].plot(kind='barh', ax=ax, color='#2f6f4e')
ax.set_title('XGBoost feature importance (habitat model)')
plt.tight_layout()
fig.savefig(OUT_DIR / 'feature_importance.png', dpi=150)
plt.show()

# SHAP on a sample (fast)
sample = X.sample(n=min(800, len(X)), random_state=42)
explainer = shap.TreeExplainer(final)
sv = explainer.shap_values(sample)
plt.figure()
shap.summary_plot(sv, sample, show=False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quick risk scatter in projected metres (no geopandas needed)
fig, ax = plt.subplots(figsize=(6, 7))
sc = ax.scatter(out['cx'], out['cy'], c=out['risk_score'], s=2, cmap='RdYlGn_r', linewidths=0)
ax.set_aspect('equal')
ax.set_title('Anuradhapura overlap risk (habitat × exposure)')
ax.set_xlabel('Easting (EPSG:5234)'); ax.set_ylabel('Northing (EPSG:5234)')
plt.colorbar(sc, ax=ax, label='risk_score')
plt.tight_layout()
fig.savefig(OUT_DIR / 'risk_scatter.png', dpi=150)
plt.show()

In [ ]:
summary = {
    'district': 'Anuradhapura',
    'n_train': int(len(train)),
    'n_presence': int((y == 1).sum()),
    'n_background': int((y == 0).sum()),
    'n_grid_cells': int(len(grid)),
    'features': FEATURE_COLS,
    'spatial_block_cells': BLOCK,
    'cv_mean_auc': float(cv['auc'].mean()),
    'cv_std_auc': float(cv['auc'].std()),
    'oof_auc': float(overall_auc),
    'oof_ap': float(overall_ap),
    'risk_formula': 'habitat_suitability * human_exposure_normalized',
}
(OUT_DIR / 'phase4_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

# Zip everything for download
zip_path = Path('/content/phase4_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.iterdir():
        zf.write(f, arcname=f.name)
print('Download this file:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Auto-download unavailable (not Colab?). Zip is at', zip_path, e)

## After Colab

1. Download `phase4_outputs.zip` (or use the auto-download cell).
2. Unzip into the project:
   - `component-1-danger-map/outputs/reports/` ← metrics, plots, `phase4_summary.json`
   - `component-1-danger-map/outputs/maps/` ← `risk_scores.parquet`, `risk_scores.csv`, `risk_scatter.png`
   - `component-1-danger-map/outputs/` ← `xgb_habitat_model.joblib`
3. Tell the local agent Phase 4 is done so we can continue Phase 5 (explanations) / map styling.